# Sentinel AI — PPE model retrain (Fix 2)

Closes the **live-feed domain gap** that causes helmet/vest false negatives.
Runs **prune negatives → bake CLAHE → train** end-to-end on a free Colab GPU.

**Before you start:** `Runtime ▸ Change runtime type ▸ GPU (T4)`.

### One-time upload
1. On your PC, zip the dataset folder `PPE_Filtered` → `PPE_Filtered.zip`.
2. Upload `PPE_Filtered.zip` to your Google Drive (top level of *My Drive*).
3. Run the cells below top to bottom.

## 0. GPU check + install

In [ ]:
!nvidia-smi -L    # confirm a GPU is attached
!pip -q install ultralytics albumentations

## 1. Get the dataset onto Colab
Mounts Drive and unzips `PPE_Filtered.zip` to `/content/PPE_Filtered`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, zipfile, time
ZIP = '/content/drive/MyDrive/PPE_Filtered.zip'   # <-- change if you put it elsewhere
DATASET = '/content/PPE_Filtered'

if not os.path.exists(DATASET):
    assert os.path.exists(ZIP), f'Not found: {ZIP} — upload PPE_Filtered.zip to My Drive'
    t=time.time()
    with zipfile.ZipFile(ZIP) as z: z.extractall('/content')
    print(f'unzipped in {time.time()-t:.0f}s')
# if the zip contained a nested folder, fix DATASET here:
print('train images:', len(os.listdir(DATASET+'/train/images')))

## 2. Prune background to 12%
Your PPE set was ~20% empty-label (background) images, which over-suppresses real
detections. This moves the excess into `backup_negatives/` (reversible).

In [ ]:
import glob, shutil, random
random.seed(42)
LBL = f'{DATASET}/train/labels'; IMG = f'{DATASET}/train/images'
BK  = f'{DATASET}/backup_negatives'; CAP = 0.12
EXT = ('.jpg','.jpeg','.png','.bmp','.webp')

def find_img(stem):
    for e in EXT:
        p=f'{IMG}/{stem}{e}'
        if os.path.exists(p): return p

labels = glob.glob(f'{LBL}/*.txt')
neg = [f for f in labels if os.path.getsize(f)==0]
pos = len(labels)-len(neg)
max_neg = int(pos*CAP/(1-CAP)); move = max(0, len(neg)-max_neg)
print(f'total={len(labels)} pos={pos} background={len(neg)} ({100*len(neg)/len(labels):.1f}%)')
print(f'moving {move} background pairs -> 12% cap')
os.makedirs(f'{BK}/labels', exist_ok=True); os.makedirs(f'{BK}/images', exist_ok=True)
for lf in random.sample(neg, move):
    stem=os.path.splitext(os.path.basename(lf))[0]; im=find_img(stem)
    shutil.move(lf, f'{BK}/labels/{os.path.basename(lf)}')
    if im: shutil.move(im, f'{BK}/images/{os.path.basename(im)}')
print('done — background now ~12%')

## 3. Bake CLAHE into the images
Applies the **exact** CLAHE (`clipLimit=2.0, tile 8x8`) your live engine uses, so
training matches inference. Writes a new dataset at `/content/PPE_Filtered_clahe`.

In [ ]:
import cv2
SRC = DATASET; DST = '/content/PPE_Filtered_clahe'
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))

def bake(p, out):
    img=cv2.imread(p)
    if img is None: return False
    lab=cv2.cvtColor(img, cv2.COLOR_BGR2LAB); l,a,b=cv2.split(lab)
    l=clahe.apply(l)
    cv2.imwrite(out, cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2BGR)); return True

for split in ['train','valid','test']:
    si=f'{SRC}/{split}/images'
    if not os.path.isdir(si): continue
    di=f'{DST}/{split}/images'; dl=f'{DST}/{split}/labels'
    os.makedirs(di, exist_ok=True); os.makedirs(dl, exist_ok=True)
    for lf in glob.glob(f'{SRC}/{split}/labels/*.txt'):
        shutil.copy2(lf, f'{dl}/{os.path.basename(lf)}')
    imgs=[p for p in glob.glob(f'{si}/*') if p.lower().endswith(EXT)]
    for i,p in enumerate(imgs):
        bake(p, f'{di}/{os.path.basename(p)}')
        if i and i%3000==0: print(f'  {split}: {i}/{len(imgs)}')
    print(f'{split}: baked {len(imgs)}')

with open(f'{DST}/data.yaml','w') as f:
    f.write('path: /content/PPE_Filtered_clahe\ntrain: train/images\nval: valid/images\ntest: test/images\n'
            'nc: 3\nnames:\n  0: Glass\n  1: Helmet\n  2: Vest\n')
print('CLAHE dataset ready at', DST)

## 4. Train
`hsv_v`/`hsv_h` are the key knobs for your dim, green-fluorescent room. ~1–3 h on a T4.

In [ ]:
from ultralytics import YOLO
model = YOLO('yolov8s.pt')
model.train(
    data=f'{DST}/data.yaml', epochs=120, imgsz=640, batch=16,
    hsv_h=0.03, hsv_s=0.7, hsv_v=0.5,       # lighting/colour for dim + fluorescent
    mosaic=1.0, mixup=0.15, degrees=5, translate=0.1, scale=0.5, erasing=0.4,
    name='ppe_v9_clahe')

## 5. Save the new weights to Drive

In [ ]:
best='/content/runs/detect/ppe_v9_clahe/weights/best.pt'
shutil.copy(best, '/content/drive/MyDrive/ppe_v9_clahe_best.pt')
print('Saved to My Drive as ppe_v9_clahe_best.pt — download it and put it in ai-engine/ on the server')

## 6. After training — on the server
1. Copy `ppe_v9_clahe_best.pt` into `ai-engine/` and point the PPE model path at it.
2. In `ai-engine/main.py`, raise the Helmet gate back from `0.12` to `~0.25`
   (the model is now confident in your lighting, so the low gate is no longer needed).
3. Restart the AI engine.